In [60]:
import pandas as pd

df_scratch = pd.read_pickle('classification_4t_scratch.pkl')
df_pretrained = pd.read_pickle('classification_2p2t_finetuned.pkl')


In [61]:
# Select all rows under "locAttn"
locattn_rows = df_scratch.xs('locAttn', level=0, drop_level=False)

# Build new MultiIndex with the new label
new_index = pd.MultiIndex.from_product(
    [['locAttn (warm start)'], locattn_rows.index.get_level_values(1)],
    names=df_scratch.index.names
)

# Reindex the rows with the new MultiIndex
locattn_rows = locattn_rows.set_index(new_index)

# Append to original df
df_scratch = pd.concat([df_scratch, locattn_rows])

In [62]:
row = df_scratch.loc[('fullAttn', '-')].copy()
row.name = ('fullAttn (warm start)', '-')
df_scratch = pd.concat([df_scratch, row.to_frame().T])

In [63]:
df_scratch.loc[('fullAttn (warm start)', '-')] = df_scratch.loc[('fullAttn', '-')]

In [64]:
df_scratch.index.names = ['Token Mixer', 'Kernel Size']
df_join = pd.merge(df_scratch, df_pretrained, on=['Token Mixer', 'Kernel Size'], suffixes=('_scratch', '_pretrained'))

In [65]:
df = pd.DataFrame()
for p in df_scratch.columns:
    df[p] = df_join[f'{p}_pretrained']/df_join[f'{p}_scratch']

df = df.loc[~df.index.get_level_values(0).str.startswith('ResNet'), df.columns.str.endswith('_auc')]
df_no_imgwoof = df.iloc[:, 1:]
df

ImageWoof_auc  PathMNIST_auc  \
Token Mixer           Kernel Size                                 
pooling               3                 1.015798       0.983398   
                      5                 1.013029       0.984896   
                      7                 1.016564       0.991963   
conv                  3                 1.020774       0.972544   
                      5                 1.022453       0.979753   
                      7                 1.040282       0.983210   
sep_conv              3                 1.011328       0.981332   
                      5                 1.015300       0.981672   
                      7                 1.052457       1.000204   
locAttn               3                 1.056709       1.004536   
                      5                 1.060413       1.001645   
                      7                 1.072332       0.995297   
fullAttn              -                 1.085650       0.999694   
identity              1                 1.027676       0.974887   
locAttn (warm start)  3                 1.062391       1.005360   
                      5                 1.087873       1.000308   
                      7                 1.100399       1.001431   
fullAttn (warm start) -                 1.116488       1.004591   

                                   DermaMNIST_auc  PneumoniaMNIST_auc  \
Token Mixer           Kernel Size                                       
pooling               3                  1.020376            0.990828   
                      5                  1.031422            1.005187   
                      7                  1.045821            0.997860   
conv                  3                  1.052104            1.007274   
                      5                  1.031282            1.005506   
                      7                  1.059689            0.995466   
sep_conv              3                  1.014999            1.002023   
                      5                  1.034301            1.005576   
                      7                  1.038360            0.998583   
locAttn               3                  1.025708            1.006366   
                      5                  1.059147            1.015849   
                      7                  1.087373            0.995104   
fullAttn              -                  0.986048            1.041504   
identity              1                  1.098133            1.002833   
locAttn (warm start)  3                  1.018153            1.007701   
                      5                  1.057859            1.013599   
                      7                  1.067392            1.009283   
fullAttn (warm start) -                  0.988337            1.044454   

                                   OrganSMNIST_auc  
Token Mixer           Kernel Size                   
pooling               3                   0.991577  
                      5                   1.025920  
                      7                   1.005839  
conv                  3                   1.001154  
                      5                   1.014142  
                      7                   1.006735  
sep_conv              3                   0.990037  
                      5                   1.014346  
                      7                   0.999788  
locAttn               3                   1.011189  
                      5                   1.031702  
                      7                   1.024448  
fullAttn              -                   0.982746  
identity              1                   0.994598  
locAttn (warm start)  3                   1.021100  
                      5                   1.018870  
                      7                   1.031954  
fullAttn (warm start) -                   0.962289

# Mean Dataset

In [66]:
df.mean(0).sort_values(ascending=False)

ImageWoof_auc         1.048773
DermaMNIST_auc        1.039806
PneumoniaMNIST_auc    1.008055
OrganSMNIST_auc       1.007135
PathMNIST_auc         0.991485
dtype: float64

# Pool Size

In [67]:
df.groupby('Kernel Size').mean()

,ImageWoof_auc,PathMNIST_auc,DermaMNIST_auc,PneumoniaMNIST_auc,OrganSMNIST_auc
Kernel Size,,,,,
-,1.101069,1.002142,0.987192,1.042979,0.972518
1,1.027676,0.974887,1.098133,1.002833,0.994598
3,1.033400,0.989434,1.026268,1.002838,1.003011
5,1.039814,0.989655,1.042802,1.009144,1.020996
7,1.056407,0.994421,1.059727,0.999259,1.013753


In [68]:
df.groupby('Kernel Size').mean().mean(1).sort_values(ascending=False)

Kernel Size
7    1.024713
-    1.021180
5    1.020482
1    1.019626
3    1.010990
dtype: float64

In [69]:
df_no_imgwoof.groupby('Kernel Size').mean().mean(1).sort_values(ascending=False)

Kernel Size
1    1.017613
7    1.016790
5    1.015649
3    1.005388
-    1.001208
dtype: float64

# Token Mixer

In [70]:
df.groupby('Token Mixer').mean()

,ImageWoof_auc,PathMNIST_auc,DermaMNIST_auc,PneumoniaMNIST_auc,OrganSMNIST_auc
Token Mixer,,,,,
conv,1.027836,0.978502,1.047692,1.002749,1.007344
fullAttn,1.085650,0.999694,0.986048,1.041504,0.982746
fullAttn (warm start),1.116488,1.004591,0.988337,1.044454,0.962289
identity,1.027676,0.974887,1.098133,1.002833,0.994598
locAttn,1.063151,1.000493,1.057409,1.005773,1.022446
locAttn (warm start),1.083555,1.002367,1.047801,1.010194,1.023974
pooling,1.015130,0.986752,1.032540,0.997958,1.007779
sep_conv,1.026361,0.987736,1.029220,1.002061,1.001390


In [71]:
df.groupby('Token Mixer').mean().mean(1).sort_values(ascending=False)

Token Mixer
locAttn (warm start)     1.033578
locAttn                  1.029854
fullAttn (warm start)    1.023232
identity                 1.019626
fullAttn                 1.019128
conv                     1.012825
sep_conv                 1.009354
pooling                  1.008032
dtype: float64

In [72]:
df_no_imgwoof.groupby('Token Mixer').mean().mean(1).sort_values(ascending=False)

Token Mixer
locAttn                  1.021530
locAttn (warm start)     1.021084
identity                 1.017613
conv                     1.009072
pooling                  1.006257
sep_conv                 1.005102
fullAttn                 1.002498
fullAttn (warm start)    0.999918
dtype: float64